## Ran Uram 206661886
## Shahar Lankry 322659137
## Daniel Geron 212515522
-------------------------
# Feature Extraction and Report

## Imports

In [342]:
import warnings
warnings.filterwarnings('ignore')
import cv2
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re
from typing import Tuple
from tqdm import tqdm
from openpyxl import load_workbook
import anthropic

## Paths and Configuration

In [343]:
normalised_images_folder =r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/FinalDataNormalized"
segment_results_folder  =r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/FinalSegmentation"
feature_tables_location =r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/FinalFeatureExtraction/tables"
COLUMN_TOKEN    = "_COLUMNS"
SIGNATURE_TOKEN = "_signature"

os.makedirs(feature_tables_location, exist_ok=True)

visualization_folder = r"C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/FinalFeatureExtraction/visualization_results"
os.makedirs(visualization_folder, exist_ok=True)
for feature in [
    'baseline', 'slant', 'stroke_thickness',
    'right_margin', 'left_margin',
    'baseline_slope',
    'top_margin', 'bottom_margin', 'column_spacing',
    'roundness', 'word_spacing', 'letter_size',
]:
    os.makedirs(os.path.join(visualization_folder, feature), exist_ok=True)

## FEATURE EXTRACTION

<table dir="rtl" style="border-collapse: collapse; width: 100%; text-align: right; border: 1px solid black;">
    <thead>
        <tr>
            <th style="border: 1px solid black; padding: 10px;">ערכי הקיצון והאמצע (סקאלה 0-1)</th>
            <th style="border: 1px solid black; padding: 10px;">מה זה אומר? (תיאור)</th>
            <th style="border: 1px solid black; padding: 10px;">שם הפיצ'ר</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = שמאלית חזקה (נגד הכיוון)<br>0.5 = אנכי לגמרי (90 מעלות, ישר)<br>1.0 = ימנית חזקה (שוכב קדימה)</td>
            <td style="border: 1px solid black; padding: 10px;">זווית הכתיבה ביחס לאנך</td>
            <td style="border: 1px solid black; padding: 10px;">נטייה (Slant)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = דקיק (קו נימי, עדין מאוד)<br>0.5 = בינוני (עובי סטנדרטי)<br>1.0 = עבה מאוד (קו "בצקי", מרוח)</td>
            <td style="border: 1px solid black; padding: 10px;">עובי הקו ביחס לגודל האות ("לחץ")</td>
            <td style="border: 1px solid black; padding: 10px;">עובי קו (Stroke Thickness)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = חותך/יורד (הכתיבה יורדת מתחת לקו)<br>0.5 = על הקו בדיוק (התאמה מושלמת)<br>1.0 = מרחף גבוה (הכתיבה מנותקת מהקו כלפי מעלה)</td>
            <td style="border: 1px solid black; padding: 10px;">מיקום האות ביחס לפס המודפס</td>
            <td style="border: 1px solid black; padding: 10px;">היצמדות לשורה (Baseline)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = צמוד לקצה הימני (0% עד 5% מרוחב הדף)<br>0.5 = מרחק מאוזן מהקצה (10% עד 15%)<br>1.0 = מרחק גדול מהקצה הימני (מעל 30%)</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק מהפיקסל השחור הראשון בשורה לקצה הימני של המסגרת</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>מדידת שוליים- ימין (Margin Right)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = צמוד לקצה השמאלי (0% עד 5% מרוחב הדף)<br>0.5 = מרחק מאוזן מהקצה (10% עד 15%)<br>1.0 = מרחק גדול מהקצה השמאלי (מעל 30%)</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק מהפיקסל האחרון לקצה השמאלי של המסגרת</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>מדידת שוליים- שמאל (Margin Left)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">-1 = זוהתה מילה אחת בלבד (לא ניתן למדוד)<br>0.0 = מילים דבוקות (אין רווח)<br>0.5 = רווח תקין בין מילים<br>1.0 = רווחים גדולים מאוד בין מילים</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק החציוני בין מילים שזוהו בשורה</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>רווח בין מילים (Word Spacing)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = כתב זעיר- תופס שטח מינימלי מהשורה<br>0.5 = כתב בינוני ותקני- פרופורציה הגיונית<br>1.0 = כתב ענק- משתלט על מרחב השורה</td>
            <td style="border: 1px solid black; padding: 10px;">המרחק האנכי של האותיות, מחושב כחציון גובה התיבות התוחמות ומנורמל לגובה הדף</td>
            <td style="border: 1px solid black; padding: 10px;"><strong>גודל הכתב האבסולוטי (Letter Size)</strong></td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = כתב זוויתי- קווים חדים ושפיצים<br>0.5 = שילוב מאוזן- עקומות מתונות וזורמות<br>1.0 = כתב עגול- צורות עגולות, רכות ומלאות</td>
            <td style="border: 1px solid black; padding: 10px;">בדיקת ה"שפיציות" של הכתב דרך חישוב מדד המעגליות של החללים הלבנים בתוך האותיות</td>
            <td style="border: 1px solid black; padding: 10px;">מעגליות מול זוויתיות (Roundness vs. Angularity)</td>
        </tr>
        <tr>
            <td style="border: 1px solid black; padding: 10px;">0.0 = שורה נופלת (שיפוע רגרסיה חיובי)<br>0.5 = שורה ישרה (קו אופקי ושטוח)<br>1.0 = שורה מטפסת (שיפוע רגרסיה שלילי)</td>
            <td style="border: 1px solid black; padding: 10px;">כיוון הזרימה של השורה, מחושב באמצעות רגרסיה לינארית על מרכזי הכובד של המילים</td>
            <td style="border: 1px solid black; padding: 10px;">שיפוע קו הבסיס (Baseline Slope)</td>
        </tr>
    </tbody>
</table>

### Slant

In [344]:
def measure_slant_by_shear(binary_img: np.ndarray) -> Tuple[float, float]:
    """
    Finds the shear angle that maximises vertical-projection variance, estimating the dominant slant direction.
    input: binary_img (np.ndarray) -- binary grayscale handwriting image
    output: (slant_score: float [0.0-1.0], optimal_angle: float in degrees)
    """
    img_height, img_width = binary_img.shape

    # Test angles from -30 to +30 degrees
    angles_to_test = np.linspace(-30, 30, 61)
    max_variance = 0
    optimal_angle = 0

    for angle in angles_to_test:
        angle_rad = np.radians(angle)
        shear_factor = np.tan(angle_rad)

        M = np.array([[1, shear_factor, 0],
                      [0, 1, 0]], dtype=np.float32)

        sheared = cv2.warpAffine(
            binary_img, M,
            (img_width + abs(int(shear_factor * img_height)), img_height),
            flags=cv2.INTER_LINEAR, borderValue=255
        )

        # Sum of black pixels per column; higher variance means more upright text
        projection = np.sum(sheared == 0, axis=0)
        variance = np.var(projection)

        if variance > max_variance:
            max_variance = variance
            optimal_angle = angle

    # Map: -30 deg -> 0.0, 0 deg -> 0.5, +30 deg -> 1.0
    slant = (optimal_angle + 30) / 60
    slant = np.clip(slant, 0.0, 1.0)
    return slant, optimal_angle

In [345]:
def measure_slant_by_moments(binary_img: np.ndarray) -> float:
    """
    Estimates slant by fitting ellipses to individual letter contours and averaging their orientation angles.
    input: binary_img (np.ndarray) -- binary grayscale handwriting image
    output: slant_score (float [0.0-1.0])
    """
    # Invert so contours are found on white objects over black background
    inverted = 255 - binary_img
    contours, _ = cv2.findContours(inverted, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return 0.5

    angles = []
    img_area = binary_img.shape[0] * binary_img.shape[1]

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < img_area * 0.0001 or area > img_area * 0.15:
            continue
        if len(contour) < 5:
            continue
        try:
            (x, y), (MA, ma), angle = cv2.fitEllipse(contour)
            if angle > 135:
                angle = angle - 180
            elif angle > 45:
                angle = 90 - angle
            angles.append(angle)
        except:
            continue

    if not angles:
        return 0.5

    # Remove outliers using Interquartile Range (IQR) filter
    angles = np.array(angles)
    q1, q3 = np.percentile(angles, [25, 75])
    iqr = q3 - q1
    if iqr > 0:
        mask = (angles >= q1 - 1.5 * iqr) & (angles <= q3 + 1.5 * iqr)
        angles = angles[mask]

    if len(angles) == 0:
        return 0.5

    mean_angle = np.clip(np.mean(angles), -20, 20)
    slant = (mean_angle + 20) / 40
    return slant

In [346]:
def extract_slant(image_path: str) -> Tuple[float, dict]:
    """
    Combines shear-based and moment-based slant estimation for a single image, returning a weighted score.
    input: image_path (str) -- path to a grayscale or binary handwriting image
    output: (slant_score: float [0.0-1.0], debug_info: dict)
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.5, {'error': 'Failed to read'}

    if len(np.unique(img)) > 2:
        _, img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    slant_shear, optimal_angle = measure_slant_by_shear(img)
    slant_moments = measure_slant_by_moments(img)

    # Shear method is more robust for full-line images; weight it higher
    slant = 0.7 * slant_shear + 0.3 * slant_moments
    slant = np.clip(slant, 0.0, 1.0)
    debug_info = {
        'slant_shear': slant_shear,
        'slant_moments': slant_moments,
        'optimal_angle': optimal_angle
    }
    return slant, debug_info

In [347]:
# process_directory (slant batch) removed -- iteration is handled in "Run All Feature Extraction"

In [348]:
# process_directory call removed

### Stroke Thickness

In [349]:
def remove_printed_text_and_lines(img: np.ndarray) -> np.ndarray:
    """
    Removes horizontal ruled lines and noise from an image, isolating handwriting strokes.
    input: img (np.ndarray) -- BGR or grayscale handwriting image
    output: cleaned (np.ndarray) -- grayscale image, black text on white background
    """
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    height, width = img.shape[:2]

    # Detect and erase horizontal ruled lines via Hough Transform
    lines_mask = np.zeros_like(binary)
    lines = cv2.HoughLinesP(binary, 1, np.pi / 180, threshold=100,
                            minLineLength=width * 0.3, maxLineGap=10)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(lines_mask, (x1, y1), (x2, y2), 255, 8)

    binary_no_lines = cv2.bitwise_and(binary, cv2.bitwise_not(lines_mask))

    # Keep only valid handwriting components; discard header, footer, and tiny noise
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_no_lines, connectivity=8)
    handwriting_mask = np.zeros_like(binary_no_lines)
    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        if not ((y + h) > height * 0.85 or y < height * 0.1 or area < 15):
            handwriting_mask[labels == i] = 255

    return cv2.bitwise_not(handwriting_mask)

In [350]:
def calculate_stroke_thickness_pure(img: np.ndarray) -> float:
    """
    Estimates average pen stroke thickness using the Distance Transform on isolated handwriting pixels.
    input: img (np.ndarray) -- BGR or grayscale handwriting image
    output: avg_thickness (float) -- raw stroke diameter in pixels (unnormalized)
    """
    handwriting_only = remove_printed_text_and_lines(img)
    gray = (cv2.cvtColor(handwriting_only, cv2.COLOR_BGR2GRAY)
            if len(handwriting_only.shape) == 3 else handwriting_only)

    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    inverted = cv2.bitwise_not(binary)

    # Each foreground pixel stores its distance to the nearest background pixel
    dist_transform = cv2.distanceTransform(inverted, cv2.DIST_L2, 5)
    text_distances = dist_transform[dist_transform > 0]

    total_pixels = inverted.shape[0] * inverted.shape[1]
    handwriting_ratio = len(text_distances) / total_pixels if total_pixels > 0 else 0

    MINIMUM_CONTENT_THRESHOLD = 0.003
    if len(text_distances) == 0 or handwriting_ratio < MINIMUM_CONTENT_THRESHOLD:
        # Fallback: apply distance transform directly to the raw (un-cleaned) image
        gray_raw = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
        _, binary_raw = cv2.threshold(gray_raw, 127, 255, cv2.THRESH_BINARY)
        inverted_raw = cv2.bitwise_not(binary_raw)
        dist_transform_raw = cv2.distanceTransform(inverted_raw, cv2.DIST_L2, 5)
        text_distances = dist_transform_raw[dist_transform_raw > 0]
        if len(text_distances) == 0:
            return 0.0

    # Mean radius * 2 = mean diameter (stroke thickness)
    return float(np.mean(text_distances) * 2)

In [351]:
# process_images_for_stroke_thickness (batch) removed -- iteration is handled in "Run All Feature Extraction"

In [352]:
# process_images_for_stroke_thickness call removed

### Baseline 

In [353]:
def extract_file_id(filename: str) -> int:
    """
    Extracts the first integer found in a filename, used as a natural sort key.
    input: filename (str)
    output: id (int) -- first integer found, or -1 if none
    """
    match = re.search(r'(\d+)', filename)
    return int(match.group(1)) if match else -1

In [354]:
def calculate_position_score(img: np.ndarray) -> Tuple[float, np.ndarray]:
    """
    Measures how handwriting sits relative to the nearest ruled line, returning a normalised position score.
    input: img (np.ndarray) -- BGR handwriting image
    output: (score: float [0.0-1.0] | None, viz_img: np.ndarray)
    """
    copy_img = img.copy()
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    img_h, img_w = thresh.shape

    # Wide horizontal kernel isolates ruled lines while ignoring text
    min_line_width = int(img_w * 0.3)
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (min_line_width, 1))
    detected_lines_map = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=1)

    cnts_lines, _ = cv2.findContours(detected_lines_map, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not cnts_lines:
        return None, copy_img

    valid_lines = [c for c in cnts_lines
                   if cv2.boundingRect(c)[1] > 10 and cv2.boundingRect(c)[1] < img_h - 10]
    if not valid_lines:
        valid_lines = [max(cnts_lines, key=cv2.contourArea)]

    text_only = cv2.subtract(thresh, detected_lines_map)
    kernel_clean = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    text_only = cv2.morphologyEx(text_only, cv2.MORPH_OPEN, kernel_clean)

    cnts_text, _ = cv2.findContours(text_only, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    text_bottoms = []
    text_centers_y = []
    for c in cnts_text:
        if cv2.contourArea(c) < 15:
            continue
        tx, ty, tw, th = cv2.boundingRect(c)
        text_bottoms.append(ty + th)
        text_centers_y.append(ty + th / 2)

    if not text_bottoms:
        return 1.0, copy_img

    # Use mean of text bottoms for baseline position
    mean_text_bottom = np.mean(text_bottoms)
    avg_text_y = np.mean(text_centers_y)

    best_line_y = 0
    min_dist = 99999
    for line_c in valid_lines:
        lx, ly, lw, lh = cv2.boundingRect(line_c)
        dist = abs(ly - avg_text_y)
        if dist < min_dist:
            min_dist = dist
            best_line_y = ly

    # Positive distance = text above the line, negative = below
    distance = best_line_y - mean_text_bottom
    SENSITIVITY = 60.0
    final_score = max(0.0, min(1.0, 0.5 + distance / SENSITIVITY))

    # Visualisation overlays
    cv2.line(copy_img, (0, int(best_line_y)), (img_w, int(best_line_y)), (255, 0, 0), 2)
    cv2.line(copy_img, (0, int(mean_text_bottom)), (img_w, int(mean_text_bottom)), (0, 0, 255), 2)
    cv2.rectangle(copy_img, (5, 5), (225, 50), (0, 0, 0), -1)
    cv2.putText(copy_img, f"Baseline: {final_score:.3f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    return final_score, copy_img

### Right Margin

In [355]:
def calculate_right_margin(img_bgr: np.ndarray, filename: str = "", vis_dir: str = None,
                           seg_img: np.ndarray = None) -> float:
    """
    Computes the right-margin score using green word boxes (lined) or red column markers (blank page).
    input: img_bgr (np.ndarray) -- BGR image, filename (str), vis_dir (str | None),
           seg_img (np.ndarray | None) -- annotated segmentation image for lined pages
    output: grade (float [0.0-1.0]), or -1.0 if detection fails
    """
    h, w = img_bgr.shape[:2]
    is_blank_page = "_COLUMNS" in filename

    if is_blank_page:
        # Blank page: rightmost column boundary from red rectangle markers
        b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]
        red_mask = (r > 150) & (g < 80) & (b < 80)
        xs = np.where(red_mask.any(axis=0))[0]
        if len(xs) == 0:
            return -1.0
        rightmost_x = int(xs[-1])
        vis_base = img_bgr

    else:
        # Lined page: rightmost edge of word bounding boxes in the segmented image
        if seg_img is None:
            return -1.0
        boxes = extract_word_boxes_from_annotated(seg_img)
        if len(boxes) == 0:
            return -1.0
        rightmost_x = max(x + bw for x, y, bw, bh in boxes)
        vis_base = seg_img

    # Distance from rightmost content to page right edge, scaled to [0.0, 1.0]
    # 0.0 = text flush with right edge, 0.5 = balanced (~15%), 1.0 = very wide (>=30%)
    margin_ratio = (w - rightmost_x) / w
    MAX_MARGIN_RATIO = 0.30
    grade = round(max(0.0, min(1.0, margin_ratio / MAX_MARGIN_RATIO)), 4)

    if vis_dir and filename:
        vis = vis_base.copy()
        overlay = vis.copy()
        # Highlight the empty right-margin area with a transparent green overlay
        cv2.rectangle(overlay, (rightmost_x, 0), (w - 1, h - 1), (0, 200, 0), -1)
        vis = cv2.addWeighted(overlay, 0.25, vis, 0.75, 0)
        cv2.line(vis, (rightmost_x, 0), (rightmost_x, h), (0, 200, 0), 2)
        cv2.rectangle(vis, (5, 5), (350, 50), (0, 0, 0), -1)
        cv2.putText(vis, f"Right Margin: {grade:.4f}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imwrite(os.path.join(vis_dir, filename), vis)

    return grade

In [356]:
def process_images_for_right_margin(
    input_dir: str,
    output_excel: str,
    vis_dir: str = None,
    seg_dir: str = None,
    extensions: tuple = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
) -> pd.DataFrame:
    """
    Iterates all images in a directory, computes the right-margin score for each, and saves results to Excel.
    input: input_dir (str), output_excel (str), vis_dir (str | None),
           seg_dir (str | None) -- folder with segmented images for lined-page detection, extensions (tuple)
    output: df (pd.DataFrame) with columns [Image_Number, Filename, Right_Margin]
    """
    image_files = set()
    for ext in extensions:
        image_files.update(Path(input_dir).glob(f'*{ext}'))
        image_files.update(Path(input_dir).glob(f'*{ext.upper()}'))

    image_files = sorted(list(image_files), key=lambda p: extract_file_id(p.name))
    total = len(image_files)

    print(f"Found {total} images to analyze for Right Margin")

    if vis_dir and not os.path.exists(vis_dir):
        os.makedirs(vis_dir)

    results = []

    for idx, img_path in enumerate(image_files, 1):
        try:
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            # Signature images carry no margin information
            if SIGNATURE_TOKEN in img_path.name:
                continue

            # Load corresponding segmented image for lined-page detection
            seg_img = None
            if seg_dir:
                seg_path = os.path.join(seg_dir, img_path.name)
                seg_img = cv2.imread(seg_path)

            grade = calculate_right_margin(img, filename=img_path.name, vis_dir=vis_dir, seg_img=seg_img)

            results.append({
                'Image_Number': extract_file_id(img_path.name),
                'Filename': img_path.name,
                'Right_Margin': grade if grade != -1.0 else None
            })

            if idx % 500 == 0:
                print(f"Processed {idx}/{total} images...")

        except Exception as e:
            print(f"[ERROR] {img_path.name}: {e}")
            results.append({
                'Image_Number': extract_file_id(img_path.name),
                'Filename': img_path.name,
                'Right_Margin': None
            })

    df = pd.DataFrame(results)
    df.to_excel(output_excel, index=False, sheet_name='Right Margin')

    wb = load_workbook(output_excel)
    ws = wb.active
    for row_num in range(2, len(results) + 2):
        ws[f'A{row_num}'] = (
            f'=VALUE(MID(B{row_num}, FIND("(",B{row_num})+1, '
            f'FIND(")",B{row_num})-FIND("(",B{row_num})-1))'
        )
    wb.save(output_excel)

    print(f"\nExcel file saved: {output_excel}")
    print("\nRight Margin Statistics (0=narrow, 1=wide):")
    print(f"Min:    {df['Right_Margin'].min():.3f}")
    print(f"Max:    {df['Right_Margin'].max():.3f}")
    print(f"Mean:   {df['Right_Margin'].mean():.3f}")

    return df

In [357]:
process_images_for_right_margin(
    input_dir=normalised_images_folder,
    output_excel=os.path.join(feature_tables_location, 'right_margin.xlsx'),
    vis_dir=os.path.join(visualization_folder, 'right_margin'),
    seg_dir=segment_results_folder
)

Found 21 images to analyze for Right Margin

Excel file saved: C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/FinalFeatureExtraction/tables\right_margin.xlsx

Right Margin Statistics (0=narrow, 1=wide):
Min:    0.020
Max:    0.635
Mean:   0.070


,Image_Number,Filename,Right_Margin
0,-1,izhak_rosenboum_COLUMNS.png,0.6355
1,1,izhak_rosenboum_line_01.png,0.0582
2,2,izhak_rosenboum_line_02.png,0.0646
3,3,izhak_rosenboum_line_03.png,0.0391
4,4,izhak_rosenboum_line_04.png,0.0374
5,5,izhak_rosenboum_line_05.png,0.0219
6,6,izhak_rosenboum_line_06.png,0.0517
7,7,izhak_rosenboum_line_07.png,0.0234
8,8,izhak_rosenboum_line_08.png,0.0679
9,9,izhak_rosenboum_line_09.png,0.0323


### Left Margin

In [358]:
def calculate_left_margin(img_bgr: np.ndarray, filename: str = "", vis_dir: str = None,
                          seg_img: np.ndarray = None) -> float:
    """
    Computes the left-margin score using green word boxes (lined) or red column markers (blank page).
    input: img_bgr (np.ndarray) -- BGR image, filename (str), vis_dir (str | None),
           seg_img (np.ndarray | None) -- annotated segmentation image for lined pages
    output: grade (float [0.0-1.0]), or -1.0 if detection fails
    """
    h, w = img_bgr.shape[:2]
    is_blank_page = "_COLUMNS" in filename

    if is_blank_page:
        # Blank page: leftmost column boundary from red rectangle markers
        b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]
        red_mask = (r > 150) & (g < 80) & (b < 80)
        xs = np.where(red_mask.any(axis=0))[0]
        if len(xs) == 0:
            return -1.0
        leftmost_x = int(xs[0])
        vis_base = img_bgr

    else:
        # Lined page: leftmost edge of word bounding boxes in the segmented image
        if seg_img is None:
            return -1.0
        boxes = extract_word_boxes_from_annotated(seg_img)
        if len(boxes) == 0:
            return -1.0
        leftmost_x = min(x for x, y, bw, bh in boxes)
        vis_base = seg_img

    # Distance from page left edge to first content, scaled to [0.0, 1.0]
    # 0.0 = text flush with left edge, 0.5 = balanced (~15%), 1.0 = very wide (>=30%)
    margin_ratio = leftmost_x / w
    MAX_MARGIN_RATIO = 0.30
    grade = round(max(0.0, min(1.0, margin_ratio / MAX_MARGIN_RATIO)), 4)

    if vis_dir and filename:
        vis = vis_base.copy()
        overlay = vis.copy()
        # Highlight the empty left-margin area with a transparent green overlay
        cv2.rectangle(overlay, (0, 0), (leftmost_x, h - 1), (0, 200, 0), -1)
        vis = cv2.addWeighted(overlay, 0.25, vis, 0.75, 0)
        cv2.line(vis, (leftmost_x, 0), (leftmost_x, h), (0, 200, 0), 2)
        cv2.rectangle(vis, (5, 5), (350, 50), (0, 0, 0), -1)
        cv2.putText(vis, f"Left Margin: {grade:.4f}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imwrite(os.path.join(vis_dir, filename), vis)

    return grade

In [359]:
def process_images_for_left_margin(
    input_dir: str,
    output_excel: str,
    vis_dir: str = None,
    seg_dir: str = None,
    extensions: tuple = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
) -> pd.DataFrame:
    """
    Iterates all images in a directory, computes the left-margin score for each, and saves results to Excel.
    input: input_dir (str), output_excel (str), vis_dir (str | None),
           seg_dir (str | None) -- folder with segmented images for lined-page detection, extensions (tuple)
    output: df (pd.DataFrame) with columns [Image_Number, Filename, Left_Margin]
    """
    image_files = set()
    for ext in extensions:
        image_files.update(Path(input_dir).glob(f'*{ext}'))
        image_files.update(Path(input_dir).glob(f'*{ext.upper()}'))

    image_files = sorted(list(image_files), key=lambda p: extract_file_id(p.name))
    total = len(image_files)

    print(f"Found {total} images to analyze for Left Margin")

    if vis_dir and not os.path.exists(vis_dir):
        os.makedirs(vis_dir)

    results = []

    for idx, img_path in enumerate(image_files, 1):
        try:
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            # Signature images carry no margin information
            if SIGNATURE_TOKEN in img_path.name:
                continue

            # Load corresponding segmented image for lined-page detection
            seg_img = None
            if seg_dir:
                seg_path = os.path.join(seg_dir, img_path.name)
                seg_img = cv2.imread(seg_path)

            grade = calculate_left_margin(img, filename=img_path.name, vis_dir=vis_dir, seg_img=seg_img)

            results.append({
                'Image_Number': extract_file_id(img_path.name),
                'Filename': img_path.name,
                'Left_Margin': grade if grade != -1.0 else None
            })

            if idx % 500 == 0:
                print(f"Processed {idx}/{total} images...")

        except Exception as e:
            print(f"[ERROR] {img_path.name}: {e}")
            results.append({
                'Image_Number': extract_file_id(img_path.name),
                'Filename': img_path.name,
                'Left_Margin': None
            })

    df = pd.DataFrame(results)
    df.to_excel(output_excel, index=False, sheet_name='Left Margin')

    wb = load_workbook(output_excel)
    ws = wb.active
    for row_num in range(2, len(results) + 2):
        ws[f'A{row_num}'] = (
            f'=VALUE(MID(B{row_num}, FIND("(",B{row_num})+1, '
            f'FIND(")",B{row_num})-FIND("(",B{row_num})-1))'
        )
    wb.save(output_excel)

    print(f"\nExcel file saved: {output_excel}")
    print("\nLeft Margin Statistics (0=narrow, 1=wide):")
    print(f"Min:    {df['Left_Margin'].min():.3f}")
    print(f"Max:    {df['Left_Margin'].max():.3f}")
    print(f"Mean:   {df['Left_Margin'].mean():.3f}")

    return df

In [360]:
process_images_for_left_margin(
    input_dir=normalised_images_folder,
    output_excel=os.path.join(feature_tables_location, 'left_margin.xlsx'),
    vis_dir=os.path.join(visualization_folder, 'left_margin'),
    seg_dir=segment_results_folder
)

Found 21 images to analyze for Left Margin

Excel file saved: C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/FinalFeatureExtraction/tables\left_margin.xlsx

Left Margin Statistics (0=narrow, 1=wide):
Min:    0.022
Max:    1.000
Mean:   0.153


,Image_Number,Filename,Left_Margin
0,-1,izhak_rosenboum_COLUMNS.png,0.7914
1,1,izhak_rosenboum_line_01.png,0.1165
2,2,izhak_rosenboum_line_02.png,0.1513
3,3,izhak_rosenboum_line_03.png,1.0000
4,4,izhak_rosenboum_line_04.png,0.0747
5,5,izhak_rosenboum_line_05.png,0.0388
6,6,izhak_rosenboum_line_06.png,0.0234
7,7,izhak_rosenboum_line_07.png,0.0853
8,8,izhak_rosenboum_line_08.png,0.1172
9,9,izhak_rosenboum_line_09.png,0.0696


### Top Margin

In [361]:
def calculate_top_margin(img_bgr: np.ndarray, filename: str = "", vis_dir: str = None) -> float:
    """
    Computes the top-margin score as the normalised y-coordinate of the highest red-marked content row.
    input: img_bgr (np.ndarray) -- BGR image with red column markers, filename (str), vis_dir (str | None)
    output: grade (float [0.0-1.0]), or -1.0 if no markers are found
    """
    h, w = img_bgr.shape[:2]
    b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]

    # Detect red pixels that mark the column/text regions from the normalisation step
    red_mask = (r > 150) & (g < 80) & (b < 80)
    ys = np.where(red_mask.any(axis=1))[0]

    if len(ys) == 0:
        return -1.0

    # Topmost row containing red markers defines the upper text boundary
    y_top = int(ys[0])
    margin_ratio = y_top / h

    # 0.0 = flush with top edge, 0.5 = balanced (~15% of height), 1.0 = very wide (>=30%)
    MAX_MARGIN_RATIO = 0.30
    score = margin_ratio / MAX_MARGIN_RATIO
    grade = round(max(0.0, min(1.0, score)), 4)

    if vis_dir and filename:
        vis = img_bgr.copy()
        overlay = vis.copy()
        # Highlight empty top-margin area with a transparent yellow-green overlay
        cv2.rectangle(overlay, (0, 0), (w, y_top), (0, 200, 200), -1)
        vis = cv2.addWeighted(overlay, 0.35, vis, 0.65, 0)
        cv2.line(vis, (0, y_top), (w, y_top), (0, 0, 255), 2)
        cv2.rectangle(vis, (5, 5), (350, 50), (0, 0, 0), -1)
        cv2.putText(vis, f"Top Margin: {grade:.4f}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imwrite(os.path.join(vis_dir, filename), vis)

    return grade

In [362]:
# process_images_for_top_margin(
#     input_dir=normalised_images_folder,
#     output_excel=os.path.join(feature_tables_location, "top_margin.xlsx"),
#     vis_dir=os.path.join(visualization_folder, "top_margin"),
# )

### Bottom Margin

In [363]:
def calculate_bottom_margin(img_bgr: np.ndarray, filename: str = "", vis_dir: str = None) -> float:
    """
    Computes the bottom-margin score as the distance from the lowest red-marked row to the page bottom.
    input: img_bgr (np.ndarray) -- BGR image with red column markers, filename (str), vis_dir (str | None)
    output: grade (float [0.0-1.0]), or -1.0 if no markers are found
    """
    h, w = img_bgr.shape[:2]
    b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]
    red_mask = (r > 150) & (g < 80) & (b < 80)
    ys = np.where(red_mask.any(axis=1))[0]
    if len(ys) == 0:
        return -1
    y_bottom = int(ys[-1])
    # Distance from last content row to page bottom, normalised against 30% threshold
    grade = round(min(1.0, ((h - 1 - y_bottom) / h) / 0.30), 4)
    if vis_dir and filename:
        vis = img_bgr.copy()
        overlay = vis.copy()
        cv2.rectangle(overlay, (0, y_bottom), (w, h), (0, 200, 200), -1)
        vis = cv2.addWeighted(overlay, 0.35, vis, 0.65, 0)
        cv2.line(vis, (0, y_bottom), (w, y_bottom), (0, 0, 255), 2)
        cv2.putText(vis, f"bottom_margin: {grade:.4f}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2)
        cv2.imwrite(os.path.join(vis_dir, filename), vis)
    return grade

In [364]:
# process_images_for_bottom_margin (batch) removed -- iteration is handled in "Run All Feature Extraction"

### Column Spacing

In [365]:
def calculate_column_spacing(img_bgr: np.ndarray, filename: str = "", vis_dir: str = None) -> float:
    """
    Measures the average horizontal gap between column bands detected from red rectangle markers.
    input: img_bgr (np.ndarray) -- BGR image with red column markers, filename (str), vis_dir (str | None)
    output: (grade (float [0.0-1.0]) or -1.0, num_columns (int)) -- tuple
    """
    h, w = img_bgr.shape[:2]
    b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]

    # Detect red column markers placed by the normalisation pipeline step
    red_mask = (r > 150) & (g < 80) & (b < 80)
    col_proj = red_mask.any(axis=0)

    bands, start = [], None
    for x, val in enumerate(col_proj):
        if val and start is None:
            start = x
        elif not val and start is not None:
            bands.append((start, x))
            start = None
    if start is not None:
        bands.append((start, w))

    num_columns = len(bands)
    if num_columns < 2:
        return -1.0, num_columns

    # Gap = start of next band minus end of current band
    gaps = [max(0, bands[i+1][0] - bands[i][1]) for i in range(len(bands) - 1)]
    avg_gap = sum(gaps) / len(gaps)
    gap_ratio = avg_gap / w

    # 0.0 = columns touching, 0.5 = normal spacing (~15% of width), 1.0 = very wide (>=30%)
    MAX_GAP_RATIO = 0.30
    score = gap_ratio / MAX_GAP_RATIO
    grade = round(max(0.0, min(1.0, score)), 4)

    if vis_dir and filename:
        vis = img_bgr.copy()
        for x1, x2 in bands:
            cv2.rectangle(vis, (x1, 0), (x2, h), (0, 200, 0), 2)
        for i in range(len(bands) - 1):
            gx1, gx2 = bands[i][1], bands[i+1][0]
            mid = h // 2
            cv2.rectangle(vis, (gx1, mid - 10), (gx2, mid + 10), (0, 0, 255), -1)
            cv2.putText(vis, f"{gaps[i]}px", (gx1 + 4, mid + 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        cv2.rectangle(vis, (5, 5), (350, 50), (0, 0, 0), -1)
        cv2.putText(vis, f"Column Spacing: {grade:.4f}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imwrite(os.path.join(vis_dir, filename), vis)

    return grade, num_columns

In [366]:
# process_images_for_column_spacing(
#     input_dir=normalised_images_folder,
#     output_excel=os.path.join(feature_tables_location, "column_spacing.xlsx"),
#     vis_dir=os.path.join(visualization_folder, "column_spacing"),
# )

### Word Spacing

In [367]:
def extract_word_boxes_from_annotated(img_bgr: np.ndarray):
    """
    Recovers word bounding boxes from the green rectangle annotations drawn by the segmentation step.
    input: img_bgr (np.ndarray) -- BGR annotated image with green word rectangles
    output: boxes (list of (x, y, w, h) tuples)
    """
    b, g, r = cv2.split(img_bgr)
    green_mask = ((g.astype(np.int32) > 150) & (r.astype(np.int32) < 80) & (b.astype(np.int32) < 80)).astype(np.uint8) * 255

    # Dilate to reconnect any broken rectangle corners
    kernel = np.ones((3, 3), np.uint8)
    green_mask = cv2.dilate(green_mask, kernel, iterations=1)

    contours, _ = cv2.findContours(green_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w >= 15 and h >= 8:
            boxes.append((x, y, w, h))

    return boxes

In [368]:
def calculate_word_spacing(img_bgr: np.ndarray):
    """
    Computes a scale-invariant raw word-spacing ratio (mean gap / average word height) for a single image.
    input: img_bgr (np.ndarray) -- BGR annotated image with green word bounding boxes from segmentation
    output: (raw_ratio: float | None, viz_img: np.ndarray)
    """
    viz_img = img_bgr.copy() if len(img_bgr.shape) == 3 else cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)

    boxes = extract_word_boxes_from_annotated(img_bgr)

    if len(boxes) == 0:
        return None, viz_img

    boxes_sorted = sorted(boxes, key=lambda b: b[0])
    avg_box_height = float(np.mean([h for (_, _, _, h) in boxes_sorted]))

    if len(boxes_sorted) == 1:
        return -1, viz_img  # Single box: inter-word spacing is undefined

    gaps = []
    overlay = viz_img.copy()
    for i in range(len(boxes_sorted) - 1):
        x_cur, y_cur, w_cur, h_cur = boxes_sorted[i]
        x_next, y_next, w_next, h_next = boxes_sorted[i + 1]
        gap = x_next - (x_cur + w_cur)
        if gap > 0:
            gaps.append(gap)
            gap_top    = min(y_cur, y_next)
            gap_bottom = max(y_cur + h_cur, y_next + h_next)
            cv2.rectangle(overlay, (x_cur + w_cur, gap_top), (x_next, gap_bottom), (200, 100, 0), -1)

    cv2.addWeighted(overlay, 0.35, viz_img, 0.65, 0, viz_img)

    if len(gaps) == 0:
        return 0.0, viz_img

    avg_gap = float(np.mean(gaps))
    raw_ratio = avg_gap / avg_box_height

    return raw_ratio, viz_img

In [369]:
# process_images_for_word_spacing (batch) removed -- iteration is handled in "Run All Feature Extraction"

In [370]:
# process_images_for_word_spacing call removed

### Letter Size 

In [371]:
def calculate_letter_size(img_bgr: np.ndarray):
    """
    Returns a scale-invariant raw letter-size ratio (mean box height / image height) for a single image.
    input: img_bgr (np.ndarray) -- BGR annotated image with green word bounding boxes from segmentation
    output: (raw_ratio: float | None, viz_img: np.ndarray)
    """
    viz_img = img_bgr.copy() if len(img_bgr.shape) == 3 else cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)
    img_height = img_bgr.shape[0]

    boxes = extract_word_boxes_from_annotated(img_bgr)

    if len(boxes) == 0:
        return None, viz_img

    heights = [h for (_, _, _, h) in boxes]
    mean_height = float(np.mean(heights))

    # Normalise by image height so the ratio is scale-invariant across different image sizes
    raw_ratio = mean_height / img_height

    overlay = viz_img.copy()
    for (x, y, w, h) in boxes:
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 200, 0), 2)
    cv2.addWeighted(overlay, 0.6, viz_img, 0.4, 0, viz_img)

    return raw_ratio, viz_img

In [372]:
# process_images_for_letter_size (batch) removed -- iteration is handled in "Run All Feature Extraction"

In [373]:
# process_images_for_letter_size call removed

## Word Aspect Ratio

In [374]:
# def calculate_dimension_raw(img_bgr: np.ndarray) -> float:
#     """
#     Computes the median height-to-width ratio of word bounding boxes from the segmented image.
#     input: img_bgr (np.ndarray) -- BGR annotated image with green word bounding boxes from segmentation
#     output: raw_ratio (float | None) -- median(h) / median(w) across all word boxes
#     """
#     boxes = extract_word_boxes_from_annotated(img_bgr)
#     if len(boxes) == 0:
#         return None

#     heights = [h for x, y, w, h in boxes]
#     widths  = [w for x, y, w, h in boxes]

#     median_h = float(np.median(heights))
#     median_w = float(np.median(widths))

#     if median_w == 0:
#         return None

#     return median_h / median_w

## Basline Slope

In [375]:
USER_MIN_COMPONENT_AREA     = 20
USER_MIN_COMPONENT_WIDTH    = 3
USER_MIN_COMPONENT_HEIGHT   = 4
USER_MIN_DENSITY            = 0.18
USER_MIN_NEIGHBOR_SUPPORT   = 2
USER_NEIGHBOR_RADIUS_X      = 25
USER_NEIGHBOR_RADIUS_Y      = 18

In [376]:
def remove_lines_for_baseline_slope(img: np.ndarray) -> np.ndarray:
    """
    Removes ruled lines using Gaussian blur, Otsu thresholding, and strict density-based component filtering.
    input: img (np.ndarray) -- BGR or grayscale handwriting image
    output: cleaned (np.ndarray) -- binary image with isolated handwriting, black text on white
    """
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    height, width = gray.shape[:2]

    lines_mask = np.zeros_like(binary)
    lines = cv2.HoughLinesP(binary, 1, np.pi / 180, threshold=100,
                            minLineLength=int(width * 0.30), maxLineGap=10)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if abs(y2 - y1) <= 3:
                cv2.line(lines_mask, (x1, y1), (x2, y2), 255, 8)

    binary_no_lines = cv2.bitwise_and(binary, cv2.bitwise_not(lines_mask))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    binary_no_lines = cv2.morphologyEx(binary_no_lines, cv2.MORPH_OPEN, kernel_open, iterations=1)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_no_lines, connectivity=8)
    handwriting_mask = np.zeros_like(binary_no_lines)

    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        density = area / max(w * h, 1)
        if not (
            (y + h) > height * 0.92
            or y < height * 0.03
            or area < USER_MIN_COMPONENT_AREA
            or w < USER_MIN_COMPONENT_WIDTH
            or h < USER_MIN_COMPONENT_HEIGHT
            or density < USER_MIN_DENSITY
        ):
            handwriting_mask[labels == i] = 255

    return cv2.bitwise_not(handwriting_mask)

In [377]:
def get_letter_centroids(img: np.ndarray):
    """
    Extracts centroids of letter-like components that have sufficient local neighbour support.
    input: img (np.ndarray) -- BGR or grayscale handwriting image
    output: letters (list of dict) sorted right-to-left with keys: x, y, w, h, cx, cy, area, density
    """
    handwriting_only = remove_lines_for_baseline_slope(img)
    gray = (cv2.cvtColor(handwriting_only, cv2.COLOR_BGR2GRAY)
            if len(handwriting_only.shape) == 3 else handwriting_only)

    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    height, width = binary.shape[:2]

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)

    candidates = []
    for i in range(1, num_labels):
        x, y, w, h, area = stats[i]
        cx, cy = centroids[i]
        density = area / max(w * h, 1)
        if (area < USER_MIN_COMPONENT_AREA or w < USER_MIN_COMPONENT_WIDTH
                or h < USER_MIN_COMPONENT_HEIGHT or density < USER_MIN_DENSITY
                or y < height * 0.03 or (y + h) > height * 0.92):
            continue
        candidates.append({'x': x, 'y': y, 'w': w, 'h': h,
                            'cx': float(cx), 'cy': float(cy),
                            'area': area, 'density': float(density)})

    letters = [
        comp for i, comp in enumerate(candidates)
        if sum(
            1 for j, other in enumerate(candidates)
            if i != j
            and abs(comp['cx'] - other['cx']) <= USER_NEIGHBOR_RADIUS_X
            and abs(comp['cy'] - other['cy']) <= USER_NEIGHBOR_RADIUS_Y
        ) >= USER_MIN_NEIGHBOR_SUPPORT
    ]

    return sorted(letters, key=lambda d: d['cx'], reverse=True)

In [378]:
def calculate_baseline_slope_raw(img_bgr: np.ndarray) -> float:
    """
    Computes the raw linear-regression slope through word bounding-box centers (RTL-aware).
    input: img_bgr (np.ndarray) -- BGR annotated image with green word bounding boxes from segmentation
    output: slope (float | None) -- regression coefficient m; positive = falling line, negative = climbing
    """
    boxes = extract_word_boxes_from_annotated(img_bgr)
    if len(boxes) < 2:
        return None

    centers_x = np.array([x + w / 2 for x, y, w, h in boxes], dtype=np.float32)
    centers_y = np.array([y + h / 2 for x, y, w, h in boxes], dtype=np.float32)

    try:
        m, _ = np.polyfit(centers_x, centers_y, 1)
    except Exception:
        return None

    return float(m)

In [379]:
# process_images_for_baseline_slope (batch) removed -- iteration is handled in "Run All Feature Extraction"

In [380]:
# process_images_for_baseline_slope call removed

# Roundness vs. Angularity

In [381]:
def calculate_angularity_raw(img: np.ndarray) -> float:
    """
    Measures angular density as the number of sharp interior pen angles per 100 perimeter pixels.
    input: img (np.ndarray) -- BGR or grayscale handwriting image
    output: angular_density (float) -- raw angularity metric; higher value = more angular handwriting
    """
    handwriting_only = remove_printed_text_and_lines(img)
    gray = (cv2.cvtColor(handwriting_only, cv2.COLOR_BGR2GRAY)
            if len(handwriting_only.shape) == 3 else handwriting_only)

    _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    inverted = cv2.bitwise_not(binary)
    contours, _ = cv2.findContours(inverted, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

    sharp_corner_count = 0
    total_perimeter = 0.0
    # 3-pixel epsilon smooths raster jaggies while preserving deliberate sharp pen angles
    epsilon_val = 3.0

    for cnt in contours:
        perimeter = cv2.arcLength(cnt, True)
        if perimeter < 20:
            continue
        total_perimeter += perimeter
        approx = cv2.approxPolyDP(cnt, epsilon_val, True)
        if len(approx) < 3:
            continue
        for i in range(len(approx)):
            p1 = approx[(i - 1) % len(approx)][0]
            p2 = approx[i][0]
            p3 = approx[(i + 1) % len(approx)][0]
            v1, v2 = p1 - p2, p3 - p2
            n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
            if n1 == 0 or n2 == 0:
                continue
            cos_theta = np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0)
            if np.degrees(np.arccos(cos_theta)) < 120:
                sharp_corner_count += 1

    if total_perimeter == 0:
        return 0.0

    return (sharp_corner_count / total_perimeter) * 100

## Run All Feature Extraction

In [382]:
def zscore_normalize(raw_values, invert=False, fallback=0.5):
    """
    Standardises a list of raw values with Z-score and maps the result to [0, 1] by clamping at +/-3 sigma.
    input: raw_values (list of float | None), invert (bool) -- flip direction, fallback (float)
    output: list of float [0.0-1.0]
    """
    valid = np.array(
        [v for v in raw_values if v is not None and not np.isnan(float(v))],
        dtype=float
    )
    if len(valid) < 2:
        return [fallback] * len(raw_values)
    mean = float(np.mean(valid))
    std  = float(np.std(valid))
    result = []
    for v in raw_values:
        if v is None:
            result.append(fallback)
            continue
        if std == 0:
            result.append(fallback)
            continue
        z = (float(v) - mean) / std
        if invert:
            z = -z
        # +/-3 sigma maps to the [0, 1] endpoints
        score = float(np.clip(0.5 + z / 6.0, 0.0, 1.0))
        result.append(round(score, 3))
    return result

In [383]:
import glob

images = sorted(
    glob.glob(os.path.join(normalised_images_folder, "*.png")) +
    glob.glob(os.path.join(normalised_images_folder, "*.jpg"))
)

blank_records = []
line_records  = []

for img_path in images:
    fname    = os.path.basename(img_path)
    img_bgr  = cv2.imread(img_path)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    print(f"Processing: {fname}")

    if COLUMN_TOKEN in fname:
        top     = calculate_top_margin(img_bgr,
                      filename=fname.replace(".png", "_top_margin.png"),
                      vis_dir=os.path.join(visualization_folder, "top_margin"))
        bottom  = calculate_bottom_margin(img_bgr,
                      filename=fname.replace(".png", "_bottom_margin.png"),
                      vis_dir=os.path.join(visualization_folder, "bottom_margin"))
        spacing, num_cols = calculate_column_spacing(img_bgr,
                      filename=fname.replace(".png", "_col_spacing.png"),
                      vis_dir=os.path.join(visualization_folder, "column_spacing"))
        print(f"  [blank] top_margin     = {top}")
        print(f"  [blank] bottom_margin  = {bottom}")
        print(f"  [blank] col_spacing    = {spacing}")
        print(f"  [blank] num_columns    = {num_cols}")
        blank_records.append({
            "Filename":       fname,
            "Top_Margin":     top,
            "Bottom_Margin":  bottom,
            "Column_Spacing": spacing,
            "Num_Columns":    num_cols,
        })

    elif SIGNATURE_TOKEN in fname:
        # Signature images: only letter size is computed
        seg_path = os.path.join(segment_results_folder, fname)
        img_seg  = cv2.imread(seg_path)
        raw_letter_size, _ = calculate_letter_size(img_seg) if img_seg is not None else (None, None)
        line_records.append({
            "Filename":              fname,
            "Slant":                 None,
            "Baseline":              None,
            "Raw_Stroke_Thickness":  None,
            "Raw_Baseline_Slope":    None,
            "Raw_Angularity":        None,
            "Raw_Word_Spacing":      None,
            "Raw_Letter_Size":       raw_letter_size,
        })

    else:
        slant_val, _        = extract_slant(img_path)
        thickness           = calculate_stroke_thickness_pure(img_bgr)
        baseline, viz_base  = calculate_position_score(img_bgr)
        if baseline is None:
            baseline = 0.5
        raw_angularity      = calculate_angularity_raw(img_gray)

        # Segmented image is needed for word spacing, letter size, and baseline slope
        seg_path = os.path.join(segment_results_folder, fname)
        img_seg  = cv2.imread(seg_path)
        if img_seg is not None:
            slope            = calculate_baseline_slope_raw(img_seg)
            raw_word_spacing, _ = calculate_word_spacing(img_seg)
            raw_letter_size, _  = calculate_letter_size(img_seg)
        else:
            slope            = None
            raw_word_spacing = None
            raw_letter_size  = None

        # Baseline viz saved immediately: score needs no z-score step
        if viz_base is not None:
            cv2.imwrite(os.path.join(visualization_folder, 'baseline', fname), viz_base)

        print(f"  [line] slant             = {slant_val:.3f}")
        print(f"  [line] stroke_thickness  = {thickness:.3f}")
        print(f"  [line] baseline          = {baseline:.3f}")
        print(f"  [line] baseline_slope    = {slope}")
        print(f"  [line] raw_angularity    = {raw_angularity:.3f}")

        line_records.append({
            "Filename":              fname,
            "Slant":                 round(float(slant_val), 3) if slant_val is not None else None,
            "Baseline":              round(float(baseline), 3),
            "Raw_Stroke_Thickness":  thickness,
            "Raw_Baseline_Slope":    slope,
            "Raw_Angularity":        raw_angularity,
            "Raw_Word_Spacing":      raw_word_spacing if raw_word_spacing not in (None, -1) else None,
            "Raw_Letter_Size":       raw_letter_size,
        })

# -- Save blank-page feature tables ------------------------------------------
if blank_records:
    df_blank = pd.DataFrame(blank_records)
    df_blank[["Filename", "Top_Margin"]].to_excel(
        os.path.join(feature_tables_location, "top_margin.xlsx"), index=False)
    df_blank[["Filename", "Bottom_Margin"]].to_excel(
        os.path.join(feature_tables_location, "bottom_margin.xlsx"), index=False)
    df_blank[["Filename", "Column_Spacing"]].to_excel(
        os.path.join(feature_tables_location, "column_spacing.xlsx"), index=False)
    df_blank[["Filename", "Num_Columns"]].to_excel(
        os.path.join(feature_tables_location, "num_columns.xlsx"), index=False)
    print(f"Saved blank-page feature tables ({len(blank_records)} rows)")

# -- Z-score normalisation + save per-feature Excel files --------------------
if line_records:
    df_line = pd.DataFrame(line_records)

    normalisation_map = [
        # (raw_column,             output_column,       invert)
        ("Raw_Stroke_Thickness",  "Stroke_Thickness",  False),
        # Baseline slope: negative m = line goes up L->R = falling in RTL reading = low score
        ("Raw_Baseline_Slope",    "Baseline_Slope",    False),
        ("Raw_Angularity",        "Roundness",         True),   # high angularity -> low roundness
        ("Raw_Word_Spacing",      "Word_Spacing",      False),
        ("Raw_Letter_Size",       "Letter_Size",       False),
    ]

    for raw_col, out_col, invert in normalisation_map:
        df_line[out_col] = zscore_normalize(df_line[raw_col].tolist(), invert=invert)

    # Signature images: only Letter_Size is meaningful; clear all other feature scores
    sig_mask = df_line["Filename"].str.contains(SIGNATURE_TOKEN, na=False)
    for col in ["Slant", "Stroke_Thickness", "Baseline",
                "Baseline_Slope", "Roundness", "Word_Spacing"]:
        df_line.loc[sig_mask, col] = None

    feature_exports = [
        ("Slant",            "slant.xlsx"),
        ("Stroke_Thickness", "stroke_thickness.xlsx"),
        ("Baseline",         "Baseline.xlsx"),
        ("Baseline_Slope",   "baseline_slope.xlsx"),
        ("Roundness",        "roundness_feature.xlsx"),
        ("Word_Spacing",     "word_spacing.xlsx"),
        ("Letter_Size",      "letter_size.xlsx"),
    ]

    for feat_col, filename in feature_exports:
        if feat_col in df_line.columns:
            df_line[["Filename", feat_col]].to_excel(
                os.path.join(feature_tables_location, filename), index=False)
            print(f"Saved {filename}")

    # -- Save all visualizations with normalized scores ----------------------
    def add_score_overlay(img, label, value):
        """Draws a black banner and writes the feature label + normalized score on the image."""
        vis = img.copy()
        cv2.rectangle(vis, (5, 5), (400, 55), (0, 0, 0), -1)
        text = f"{label}: {value:.3f}" if value is not None else f"{label}: N/A"
        cv2.putText(vis, text, (10, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
        return vis

    print("\nSaving visualizations with normalized scores...")
    for _, row in df_line.iterrows():
        fname    = row['Filename']
        img_bgr  = cv2.imread(os.path.join(normalised_images_folder, fname))
        if img_bgr is None:
            continue

        # Signature images: only save letter size visualization
        if SIGNATURE_TOKEN in fname:
            seg_path = os.path.join(segment_results_folder, fname)
            img_seg  = cv2.imread(seg_path)
            if img_seg is not None:
                _, viz_ls = calculate_letter_size(img_seg)
                cv2.imwrite(os.path.join(visualization_folder, 'letter_size', fname),
                            add_score_overlay(viz_ls, 'Letter Size', row.get('Letter_Size')))
            continue

        img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

        # Slant
        cv2.imwrite(os.path.join(visualization_folder, 'slant', fname),
                    add_score_overlay(img_bgr, 'Slant', row.get('Slant')))

        # Stroke Thickness
        cv2.imwrite(os.path.join(visualization_folder, 'stroke_thickness', fname),
                    add_score_overlay(img_bgr, 'Stroke Thickness', row.get('Stroke_Thickness')))


        # Roundness
        cv2.imwrite(os.path.join(visualization_folder, 'roundness', fname),
                    add_score_overlay(img_bgr, 'Roundness', row.get('Roundness')))

        # Baseline Slope -- regression line through word-box centers on the segmented image
        seg_path = os.path.join(segment_results_folder, fname)
        img_seg  = cv2.imread(seg_path)
        if img_seg is not None:
            boxes = extract_word_boxes_from_annotated(img_seg)
            vis_slope = img_seg.copy()
            if len(boxes) >= 2:
                centers_x = np.array([x + w / 2 for x, y, w, h in boxes], dtype=np.float32)
                centers_y = np.array([y + h / 2 for x, y, w, h in boxes], dtype=np.float32)
                m, b_coef = np.polyfit(centers_x, centers_y, 1)
                x_min, x_max = int(np.min(centers_x)), int(np.max(centers_x))
                pt1 = (x_min, int(m * x_min + b_coef))
                pt2 = (x_max, int(m * x_max + b_coef))
                cv2.line(vis_slope, pt1, pt2, (0, 0, 255), 2)
                for x, y, w, h in boxes:
                    cv2.circle(vis_slope, (int(x + w / 2), int(y + h / 2)), 4, (255, 0, 0), -1)
            vis_slope = add_score_overlay(vis_slope, 'Baseline Slope', row.get('Baseline_Slope'))
            cv2.imwrite(os.path.join(visualization_folder, 'baseline_slope', fname), vis_slope)


            # Word Spacing
            _, viz_ws = calculate_word_spacing(img_seg)
            cv2.imwrite(os.path.join(visualization_folder, 'word_spacing', fname),
                        add_score_overlay(viz_ws, 'Word Spacing', row.get('Word_Spacing')))

            # Letter Size
            _, viz_ls = calculate_letter_size(img_seg)
            cv2.imwrite(os.path.join(visualization_folder, 'letter_size', fname),
                        add_score_overlay(viz_ls, 'Letter Size', row.get('Letter_Size')))

    print("Visualizations saved.")

Processing: izhak_rosenboum_COLUMNS.png
  [blank] top_margin     = 0.2187
  [blank] bottom_margin  = 0.9444
  [blank] col_spacing    = 0.6551
  [blank] num_columns    = 3
Processing: izhak_rosenboum_line_01.png
  [line] slant             = 0.532
  [line] stroke_thickness  = 2.697
  [line] baseline          = 0.650
  [line] baseline_slope    = 0.007114258207493098
  [line] raw_angularity    = 4.006
Processing: izhak_rosenboum_line_02.png
  [line] slant             = 0.783
  [line] stroke_thickness  = 2.809
  [line] baseline          = 0.628
  [line] baseline_slope    = 0.005061319510710509
  [line] raw_angularity    = 3.918
Processing: izhak_rosenboum_line_03.png
  [line] slant             = 0.823
  [line] stroke_thickness  = 2.849
  [line] baseline          = 0.856
  [line] baseline_slope    = -0.006279434850863415
  [line] raw_angularity    = 4.106
Processing: izhak_rosenboum_line_04.png
  [line] slant             = 0.528
  [line] stroke_thickness  = 2.876
  [line] baseline          =

## Unified Feature Table

Merge all per-feature Excel files into a single table — one row per image, one column per feature.

In [384]:
# Feature file registry: (excel_filename, column_in_file, output_column_name)
feature_registry = [
    # Blank-page features (COLUMN images only)
    ('top_margin.xlsx',        'Top_Margin',       'Top_Margin'),
    ('bottom_margin.xlsx',     'Bottom_Margin',     'Bottom_Margin'),
    ('column_spacing.xlsx',    'Column_Spacing',    'Column_Spacing'),
    # Margin features (all lined images via batch runner)
    ('right_margin.xlsx',      'Right_Margin',      'Right_Margin'),
    ('left_margin.xlsx',       'Left_Margin',       'Left_Margin'),
    # Line-image features (from pipeline)
    ('slant.xlsx',             'Slant',             'Slant'),
    ('stroke_thickness.xlsx',  'Stroke_Thickness',  'Stroke_Thickness'),
    ('Baseline.xlsx',          'Baseline',          'Baseline'),
    ('baseline_slope.xlsx',    'Baseline_Slope',    'Baseline_Slope'),
    ('roundness_feature.xlsx', 'Roundness',         'Roundness'),
    ('word_spacing.xlsx',      'Word_Spacing',      'Word_Spacing'),
    ('letter_size.xlsx',       'Letter_Size',       'Letter_Size'),
]

merged = None
for excel_file, src_col, out_col in feature_registry:
    path = os.path.join(feature_tables_location, excel_file)
    if not os.path.exists(path):
        print(f"[SKIP] {excel_file} not found")
        continue
    df = pd.read_excel(path)
    if 'Filename' not in df.columns or src_col not in df.columns:
        print(f"[SKIP] {excel_file}: expected columns Filename + {src_col}, got {list(df.columns)}")
        continue
    df = df[['Filename', src_col]].rename(columns={src_col: out_col})
    df = df.groupby('Filename', as_index=False)[out_col].mean()
    merged = df if merged is None else merged.merge(df, on='Filename', how='outer')

# Fill every missing cell with -1 (feature not applicable to that image type)
feature_cols = [c for c in merged.columns if c != 'Filename']
merged[feature_cols] = merged[feature_cols].fillna(-1).round(4)

# Sort by numeric ID embedded in filename (e.g. "txt (42).png" -> 42); keep as display order only
def _sort_key(fn):
    m = re.search(r'\((\d+)\)', str(fn))
    return int(m.group(1)) if m else -1

merged['_sort'] = merged['Filename'].apply(_sort_key)
merged.sort_values('_sort', inplace=True)
merged.drop(columns=['_sort'], inplace=True)
merged.reset_index(drop=True, inplace=True)

# Average row: -1 means "not applicable" so exclude it from the mean
avg_values = {col: merged[col].replace(-1, np.nan).mean() for col in feature_cols}
avg_row = pd.DataFrame([{'Filename': 'Average', **avg_values}])
merged = pd.concat([merged, avg_row], ignore_index=True)

# Append Number_of_Columns summary row (one per COLUMNS image)
if blank_records:
    for _rec in blank_records:
        _ncols = int(_rec.get("Num_Columns", 0))
        if _ncols > 0:
            _fname = _rec["Filename"]
            _label = f"Number_of_Columns ({_fname})"
            _row = pd.DataFrame([{"Filename": _label, "Number_of_Columns": _ncols}])
            merged = pd.concat([merged, _row], ignore_index=True)

output_path = os.path.join(feature_tables_location, 'all_features.xlsx')
merged.to_excel(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {merged.shape[0]-1} images x {len(feature_cols)} features (+1 average row)")
print(merged.head(3).to_string())

Saved: C:/Users/ranor/Desktop/ocr/Handwriting-OCR-for-Graphology/FinalFeatureExtraction/tables\all_features.xlsx
Shape: 22 images x 12 features (+1 average row)
                      Filename  Top_Margin  Bottom_Margin  Column_Spacing  Right_Margin  Left_Margin  Slant  Stroke_Thickness  Baseline  Baseline_Slope  Roundness  Word_Spacing  Letter_Size  Number_of_Columns
0  izhak_rosenboum_COLUMNS.png      0.2187         0.9444          0.6551        0.6355       0.7914 -1.000            -1.000    -1.000          -1.000     -1.000        -1.000       -1.000                NaN
1  izhak_rosenboum_line_01.png     -1.0000        -1.0000         -1.0000        0.0582       0.1165  0.532             0.222     0.650           0.626      0.371         0.337        0.401                NaN
2  izhak_rosenboum_line_02.png     -1.0000        -1.0000         -1.0000        0.0646       0.1513  0.783             0.484     0.628           0.572      0.413         0.619        0.376                NaN


## Graphological Report

Send the unified feature table to Claude Opus and receive a full personal graphological report.

In [386]:
%pip install weasyprint

   ---------------------------------------- 0.0/322.9 kB ? eta -:--:--
   --- ------------------------------------ 30.7/322.9 kB 1.3 MB/s eta 0:00:01
   ----------- ---------------------------- 92.2/322.9 kB 1.3 MB/s eta 0:00:01
   ------------------------ --------------- 194.6/322.9 kB 1.7 MB/s eta 0:00:01
   ---------------------------------------  317.4/322.9 kB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 322.9/322.9 kB 1.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------ --------------------------------- 0.3/2.1 MB 21.8 MB/s eta 0:00:01
   ---------------- ----------------------- 0.8/2.1 MB 13.4 MB/s eta 0:00:01
   --------------------------------- ------ 1.8/2.1 MB 13.9 MB/s eta 0:00:01
   ---------------------------------------  2.1/2.1 MB 12.1 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 11.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/369.0 kB ? eta -:--:--
   ---


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
_api_key =

# ── 1. Load the unified feature table ──────────────────────────────────────
df = pd.read_excel(os.path.join(feature_tables_location, 'all_features.xlsx'))
avg = df[df['Filename'] == 'Average'].iloc[0]

# Detect line count (exclude signature, columns, average, summary rows)
reg_rows = df[~df['Filename'].astype(str).str.contains(
    SIGNATURE_TOKEN + r'|_COLUMNS|Average|Number_of_Columns', na=False, regex=True)]
line_count = len(reg_rows)

# Detect number of columns from summary row
_ncols_rows = df[df['Filename'].astype(str).str.startswith('Number_of_Columns')]
num_cols = int(_ncols_rows['Number_of_Columns'].iloc[0]) if not _ncols_rows.empty else None

# Separate signature vs. regular rows for Letter_Size
sig_rows = df[df['Filename'].astype(str).str.contains(SIGNATURE_TOKEN, na=False)]

_sig_ls = sig_rows['Letter_Size'].replace(-1, float('nan')).dropna()
sig_letter_size = float(_sig_ls.mean()) if not _sig_ls.empty else -1

_reg_ls = reg_rows['Letter_Size'].replace(-1, float('nan')).dropna()
reg_letter_size = float(_reg_ls.mean()) if not _reg_ls.empty else -1

# ── 2. Build the feature text ──────────────────────────────────────────────
feature_meta = {
    'Baseline':       ('היצמדות לשורה – אזור כתיבה',
                       '0=אזור תחתון, 0.5=על השורה, 1=אזור עליון'),
    'Left_Margin':    ('שולי שמאל – שולי סיום',
                       '0=ללא שוליים (0-5%), 0.5=תקני ~2 ס"מ, 1=רחב מאוד (>30%)'),
    'Right_Margin':   ('שולי ימין – שולי פתיחה',
                       '0=ללא שוליים (0-5%), 0.5=תקני ~2 ס"מ, 1=רחב מאוד (>30%)'),
    'Word_Spacing':   ('מרווח בין מילים',
                       '0=צפוף, 0.5=תקני ~2 אותיות, 1=מרווח עצום'),
    'Slant':          ('נטיית הכתב',
                       '0=שמאלה חזקה, 0.5=אנכי, 1=ימינה חזקה'),
    'Roundness':      ('מעגליות מול זוויתיות',
                       '0=זוויתי מאוד, 0.5=תקני, 1=עגול מאוד'),
    'Column_Spacing': ('מרווח בין טורים',
                       '0=טורים צמודים/חופפים, 0.5=מרחק תקין, 1=מרווח עצום'),
    'Top_Margin':     ('שול עליון',
                       '0=צמוד לקצה (0-5%), 0.5=תקני 10-15%, 1=מרחק גדול (>30%)'),
    'Bottom_Margin':  ('שול תחתון',
                       '0=צמוד לקצה (0-5%), 0.5=תקני 10-15%, 1=מרחק גדול (>30%)'),
}

feature_lines = [
    f"• {desc}  [{scale}]  →  {float(avg[col]):.3f}"
    for col, (desc, scale) in feature_meta.items()
    if col in avg.index and pd.notna(avg[col]) and float(avg[col]) != -1
]

# Combined Stroke_Thickness + Baseline_Slope entry
_st = float(avg['Stroke_Thickness']) if 'Stroke_Thickness' in avg.index and pd.notna(avg['Stroke_Thickness']) and float(avg['Stroke_Thickness']) != -1 else None
_bs = float(avg['Baseline_Slope']) if 'Baseline_Slope' in avg.index and pd.notna(avg['Baseline_Slope']) and float(avg['Baseline_Slope']) != -1 else None
if _st is not None and _bs is not None:
    feature_lines.append(
        f'• עובי קו ושיפוע קו הבסיס  '
        f'[עובי: 0=דקיק ועדין, 0.5=בינוני ותקני, 1=עבה ובצקי | '
        f'שיפוע: 0=שורה יורדת, 0.5=שורה ישרה, 1=שורה עולה]  '
        f'→  עובי: {_st:.3f},  שיפוע: {_bs:.3f}'
    )
elif _st is not None:
    feature_lines.append(
        f'• עובי קו  [0=דקיק ועדין, 0.5=בינוני ותקני, 1=עבה ובצקי]  →  {_st:.3f}'
    )
elif _bs is not None:
    feature_lines.append(
        f'• שיפוע קו הבסיס  [0=שורה יורדת, 0.5=שורה ישרה, 1=שורה עולה]  →  {_bs:.3f}'
    )

if reg_letter_size != -1:
    feature_lines.append(
        f'• גודל כתב יד רגיל  [0=זעיר, 0.5=בינוני ~3 מ"מ, 1=ענק]  →  {reg_letter_size:.3f}'
    )

if sig_letter_size != -1:
    feature_lines.append(
        f'• גודל חתימה ביחס לכתב  [0=חתימה קטנה (צניעות/פרטיות), 0.5=כגודל הכתב (עקביות), 1=חתימה גדולה (נוכחות ציבורית)]  →  {sig_letter_size:.3f}'
    )

features_text = '\n'.join(feature_lines)

# ── 3. Disclaimers ─────────────────────────────────────────────────────────
column_notice = ''
if num_cols is not None and num_cols != 3:
    column_notice = (
        f'⚠️ הערת מערכת: מספר הטורים שזוהה בדף הוא {num_cols} (במקום 3 כנדרש). '
        'אין להסתמך על פיצ\u05f3רים הקשורים לטורים (מרווח בין טורים) בניתוח זה. '
        'יש להתייחס לחריגה זו בדוח ולציין שהנתון אינו מהימן.\n\n'
    )

data_notice = ''
if 10 <= line_count <= 15:
    data_notice = (
        f'⚠️ הסתייגות: הניתוח מבוסס על {line_count} שורות כתיבה בלבד. '
        'מספר שורות זה נמוך מהאידיאלי (16+), ולכן מידת הדיוק עשויה להיות מוגבלת. '
        'יש לציין הסתייגות זו בתוך הדוח.\n\n'
    )

# ── 4. Graphology dictionary ────────────────────────────────────────────────
graphology_dict = (
    'מילון הפיצ\u05f3רים הגרפולוגיים המלא\n'
    '\n'
    '1. נטיית הכתב (Slant)\n'
    'נטיית הכתב חושפת את היחס של הכותב לסביבה ואת המזג שלו.\n'
    '• 0.0 (שמאלית חזקה): מעידה על התנגדות, מופנמות, היצמדות לעבר ולעיתים מרדנות.\n'
    '• 0.5 (אנכי לגמרי): משקף היגיון, שליטה עצמית, קור רוח וגישה רציונלית לחיים.\n'
    '• 1.0 (ימנית חזקה): מצביעה על מוחצנות, ספונטניות, רגשנות וצורך עז להתקדם לעבר העתיד.\n'
    '\n'
    '2. עובי קו (Stroke Thickness)\n'
    'עובי הקו מייצג את לחץ הכתיבה, שמעיד על כמות האנרגיה והיצרים של הכותב.\n'
    '• 0.0 (דקיק ועדין): מעיד על רגישות גבוהה, רוחניות, עדינות, ולעיתים חוסר באנרגיה פיזית.\n'
    '• 0.5 (בינוני ותקני): משקף חיוניות מאוזנת ובריאה.\n'
    '• 1.0 (עבה ובצקי): מצביע על יצריות חזקה, נהנתנות, חומריות ואנרגיה פיזית מתפרצת.\n'
    '\n'
    '3. היצמדות לשורה (Baseline Alignment)\n'
    'האופן שבו הכותב ממקם את האותיות ביחס לשורה משקף את רמת היציבות והחיבור שלו למציאות.\n'
    '• 0.0 (מתחת לשורה – אזור תחתון): מעיד על עייפות, כבדות, גישה פסימית או חיבור חזק לחומר.\n'
    '• 0.5 (על הקו בדיוק – אזור אמצע): מצביע על יציבות רגשית, משמעת וראייה ריאליסטית.\n'
    '• 1.0 (מרחף מעל השורה – אזור עליון): משקף אופטימיות, רוחניות, דמיון מפותח.\n'
    '\n'
    '4. שיפוע קו הבסיס (Baseline Slope)\n'
    'הכיוון הכללי של השורה חושף את רמת האנרגיה ומצב הרוח של הכותב.\n'
    '• 0.0 (שורה יורדת): מעידה על פסימיות, אכזבה, עייפות וחוסר אנרגיה לסיים משימות.\n'
    '• 0.5 (שורה ישרה ושטוחה): משקפת יציבות רגשית, קור רוח, ראייה ריאליסטית והתמדה.\n'
    '• 1.0 (שורה עולה): מצביעה על אופטימיות, שאפתנות, התלהבות ואנרגיה מתפרצת.\n'
    '\n'
    '5. מרווח בין מילים (Word Spacing)\n'
    'משקף את המרחק החברתי שהכותב לוקח מהסביבה שלו.\n'
    '• 0.0 (צפוף ודבוק): מצביע על צורך עז בקרבה, חוסר יכולת להציב גבולות ותלות חברתית.\n'
    '• 0.5 (מרווח תקני – גודל של 2 אותיות): מעיד על תקשורת בריאה ומרחק חברתי מאוזן.\n'
    '• 1.0 (מרווח עצום): משקף צורך עז בעצמאות, בדידות, ריחוק רגשי.\n'
    '\n'
    '6. גודל כתב יד רגיל (Letter Size)\n'
    'גודל הכתב מלמד על תחושת הערך העצמי ועל המקום שהכותב רוצה לתפוס בעולם.\n'
    '• 0.0 (כתב זעיר): מעיד על יכולת ריכוז, ירידה לפרטים, מופנמות, צניעות.\n'
    '• 0.5 (כתב בינוני – ~3 מ"מ לאות): משקף אגו מאוזן והסתגלות בריאה למציאות.\n'
    '• 1.0 (כתב ענק): מצביע על מוחצנות, שאפתנות, צורך בתשומת לב.\n'
    '\n'
    '7. מעגליות מול זוויתיות (Roundness)\n'
    'הצורה הגיאומטרית חושפת את מידת הרכות או הנוקשות של הכותב.\n'
    '• 0.0 (כתב זוויתי וחד): מעיד על חשיבה אנליטית, תוקפנות, עקשנות, נחישות.\n'
    '• 0.5 (כתב תקני ומאוזן): משקף גמישות מחשבתית.\n'
    '• 1.0 (כתב עגול ורך): מצביע על רכות, פשרנות, רגשנות, חברותיות.\n'
    '\n'
    '8. שולי ימין – שולי פתיחה (Right Margin)\n'
    'שולי ימין מייצגים את היחס לעבר, למשפחה ולנקודת המוצא.\n'
    '• 0.0 (צמוד לקצה – 0-5%): מעיד על היצמדות לעבר, קושי לשחרר, חסכנות.\n'
    '• 0.5 (מאוזן ותקני – ~2 ס"מ): משקף סדר פנימי ואיזון בין העבר להווה.\n'
    '• 1.0 (מרחק גדול – מעל 30%): מצביע על רצון לברוח מהעבר וצורך בעצמאות.\n'
    '\n'
    '9. שולי שמאל – שולי סיום (Left Margin)\n'
    'שולי שמאל מייצגים את היחס לעתיד, לחברה ולעולם שבחוץ.\n'
    '• 0.0 (צמוד לקצה – 0-5%): מעיד על ריצה אימפולסיבית קדימה, חוסר מעצורים.\n'
    '• 0.5 (מאוזן ותקני – ~2 ס"מ): משקף משמעת עצמית והסתגלות חברתית תקינה.\n'
    '• 1.0 (מרחק גדול – מעל 30%): מצביע על רתיעה מהעתיד, ביישנות, פחד מכישלון.\n'
    '\n'
    '10. מרווח בין טורים (Column Spacing)\n'
    'מייצג את יכולת הארגון וחלוקת המשאבים של הכותב.\n'
    '• 0.0 (צפוף – טורים צמודים): מעיד על חוסר תכנון מראש, עומס פנימי, חרדה.\n'
    '• 0.5 (מרחק תקין ואחיד): משקף כושר ארגון מעולה וחשיבה צלולה.\n'
    '• 1.0 (מרווח עצום): מצביע על צורך קיצוני בהפרדה ובפרטיות.\n'
    '\n'
    '11. שול עליון (Top Margin)\n'
    'מייצג את הגישה של הכותב כלפי סמכות ומוסכמות.\n'
    '• 0.0 (צמוד לקצה – 0-5%): משקף דחף חזק לפעולה מהירה, חוסר כבוד לסמכות.\n'
    '• 0.5 (מאוזן ותקני – 10-15%): מעיד על דרך ארץ, נימוס, ריסון עצמי בריא.\n'
    '• 1.0 (מרחק גדול – מעל 30%): מצביע על היסוס רב, ביישנות, פחד מהתחלות חדשות.\n'
    '\n'
    '12. שול תחתון (Bottom Margin)\n'
    'מייצג את אופן סיום המשימות ויכולת ויסות האנרגיה.\n'
    '• 0.0 (צמוד לקצה – 0-5%): מעיד על רצון לנצל משאבים עד תומם, מעשיות.\n'
    '• 0.5 (מאוזן ותקני – 10-15%): משקף יכולת סיום טובה ומבוקרת.\n'
    '• 1.0 (מרחק גדול – מעל 30%): מצביע על קושי לסיים משימות ובריחה מאחריות.\n'
    '\n'
    'גודל חתימה (Signature Letter Size)\n'
    'גודל החתימה ביחס לכתב היד מגלה את הפער בין הדמות הציבורית לפרטית.\n'
    '• 0.0 (חתימה קטנה): מעידה על צניעות, ביישנות, רצון להצטמצם בעולם החיצוני.\n'
    '• 0.5 (חתימה בגודל הכתב): משקפת עקביות בין הדמות הפנימית לחיצונית, יושרה.\n'
    '• 1.0 (חתימה גדולה מהכתב): מצביעה על רצון חזק בנוכחות ציבורית ושאפתנות.\n'
)

# ── 5. Call Claude API ──────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=_api_key)

system_prompt = (
    'אתה גרפולוג ותיק ומנוסה, שכתב דוחות אישיים לאלפי אנשים לאורך עשרות שנות עבודה.\n'
    'הסגנון שלך: פסיכולוגי, חם, ישיר ובטוח בעצמו.\n'
    '\n'
    'כללים שאסור לשבור:\n'
    '- כתוב עברית טבעית, ספרותית וזורמת\n'
    '- אל תכלול ביטויי מחשב כמו "מן הנתונים עולה", "על סמך הניתוח", "ניכר כי"\n'
    '- אל תרשום רשימות תבליטים – רק פסקאות נרטיביות\n'
    '- כתוב בגוף שלישי: "הכותב" / "הכותבת" (בחר אחד ועקוב בעקביות)\n'
    '- אל תציג את עצמך ואל תוסיף הקדמות – פתח ישר בתיאור האדם\n'
    '- הביטחון העצמי שלך בא לידי ביטוי בטענות ישירות, לא בהסתייגויות\n'
    '\n'
    'מבנה הדוח (חובה לעקוב בדיוק):\n'
    'כותרת ראשית: "דוח גרפולוגי אישי"\n'
    'קטגוריה 1: היחס לזמן ולמרחב\n'
    'קטגוריה 2: יציבות פנימית ומצב רוח\n'
    'קטגוריה 3: הדינמיקה החברתית\n'
    'קטגוריה 4: קצב העבודה והתמדה\n'
    'קטגוריה 5: הדמות הפנימית והציבורית\n'
    '\n'
    'כל קטגוריה חייבת להכיל בדיוק שתי פסקאות – לא יותר, לא פחות.\n'
    'הדוח כולו לא יעלה על עמוד וחצי (כ-600–750 מילים לכלל הגוף).\n'
    'אם הקטגוריות ארוכות מדי, קצר כל אחת – שמור על שתי פסקאות לכל קטגוריה.\n'
    '\n'
    'סיום חובה:\n'
    'לאחר כל הקטגוריות, הוסף משפט סיכום חיובי ומעודד אחד המסכם את הפוטנציאל של הכותב.\n'
    'לאחר משפט הסיכום, הוסף את המשפט הבא בדיוק (ללא שינוי):\n'
    '"אנחנו מאחלים לך בהצלחה בהמשך הדרך, ומקווים למימוש כל יכולותיך ובקשותיך"\n'
    '\n'
    'כיצד לטפל בסתירות בין פיצ\u05f3רים:\n'
    'כאשר שני פיצ\u05f3רים מצביעים בכיוונים מנוגדים, עדף את הפיצ\u05f3ר העמוק (נטייה, עובי קו, שיפוע שורה) '
    'כשורש ואת האחר כביטוי חיצוני. תאר מתח פנימי כחלק אינטגרלי מהאישיות.\n'
)

user_prompt = (
    f'{column_notice}'
    f'{data_notice}'
    'להלן נתוני כתב יד מנורמלים לסקלה 0–1:\n'
    '\n'
    f'{features_text}\n'
    '\n'
    'הערה: גודל כתב יד רגיל וגודל חתימה ביחס לכתב הם שני ממדים שונים לחלוטין.\n'
    'גודל הכתב הרגיל – חושף את תחושת הערך העצמי הפנימית.\n'
    'גודל החתימה ביחס לכתב – חושף את הפער בין הדמות הפרטית לציבורית.\n'
    '\n'
    'מילון הפירושים הגרפולוגי:\n'
    f'{graphology_dict}\n'
    '\n'
    'כתוב דוח גרפולוגי אישי לפי המבנה שנקבע בהנחיות המערכת.\n'
    'שים לב לסתירות בין הפיצ\u05f3רים וטפל בהן כמתואר – אל תדלג עליהן.\n'
)

response = client.messages.create(
    model='claude-opus-4-5',
    max_tokens=2048,
    system=system_prompt,
    messages=[{'role': 'user', 'content': user_prompt}]
)

report = response.content[0].text
print(report)

# ── 6. Save report as HTML and PDF ─────────────────────────────────────────
import html as _html_mod

_report_escaped = _html_mod.escape(report).replace('\n', '<br>\n')
_html = (
    '<!DOCTYPE html>\n'
    '<html dir="rtl" lang="he">\n'
    '<head>\n'
    '<meta charset="UTF-8">\n'
    '<style>\n'
    '  body { font-family: Arial, sans-serif; direction: rtl; padding: 40px; '
    'max-width: 800px; margin: auto; line-height: 1.8; color: #222; }\n'
    '  h1 { text-align: center; font-size: 22px; margin-bottom: 30px; }\n'
    '  p { margin-bottom: 16px; }\n'
    '</style>\n'
    '</head>\n'
    '<body>\n'
    f'{_report_escaped}\n'
    '</body>\n'
    '</html>\n'
)

_html_path = os.path.join(feature_tables_location, 'graphological_report.html')
with open(_html_path, 'w', encoding='utf-8') as _f:
    _f.write(_html)
print(f'HTML report saved to {_html_path}')
_pdf_path = os.path.join(feature_tables_location, 'graphological_report.pdf')
_pdf_saved = False

try:
    import weasyprint as _wp
    _wp.HTML(string=_html).write_pdf(_pdf_path)
    print(f'PDF report saved to {_pdf_path}')
    _pdf_saved = True
except (ImportError, OSError):
    pass

if not _pdf_saved:
    try:
        import pdfkit as _pdfkit
        _pdfkit_opts = {'encoding': 'UTF-8', 'enable-local-file-access': None}
        _pdfkit.from_string(_html, _pdf_path, options=_pdfkit_opts)
        print(f'PDF report saved to {_pdf_path}')
        _pdf_saved = True
    except (ImportError, OSError):
        pass

if not _pdf_saved:
    print(f'PDF generation unavailable on this system.')
    print(f'HTML report saved to {_html_path}')
    print('To get a PDF: open the HTML in Chrome/Edge and press Ctrl+P → Save as PDF')
    print('Or install wkhtmltopdf from https://wkhtmltopdf.org and run: pip install pdfkit')


# דוח גרפולוגי אישי

## היחס לזמן ולמרחב

הכותב מציג דפוס מעניין ביחסו לזמן ולמרחב הפיזי של הדף. השוליים הימניים הצרים מאוד חושפים אדם שנשאר מחובר לשורשיו, למשפחתו ולעברו – לא מתוך כבילות, אלא מתוך תחושת ביטחון שהיא מעגן רגשי יציב. מנגד, השוליים השמאליים הצרים אף הם מעידים על אדם שאינו מהסס לנוע קדימה, שאינו חושש מהעתיד ומוכן לפגוש אותו בחזית מלאה.

המרווח הרחב שהכותב משאיר בתחתית הדף מספר סיפור נוסף. יש כאן נטייה להשאיר עניינים פתוחים, להימנע מסגירת מעגלים מוחלטת – אולי מתוך תחושה שתמיד יש עוד מה להוסיף, שהמילה האחרונה מעולם לא נאמרה. השול העליון הצר יחסית מגלה אדם שנכנס למשימות בלי היסוסים מיותרים, שאינו זקוק לטקסי פתיחה ארוכים כדי להתחיל לפעול.

## יציבות פנימית ומצב רוח

הכתיבה המרחפת מעט מעל השורה חושפת אדם בעל נטייה טבעית לאופטימיות עדינה. הכותב אינו שקוע בחומר הגולמי של החיים, אלא מרים את מבטו מעט כלפי מעלה – לא בבריחה מהמציאות, אלא בחיפוש אחר משמעות ואפשרויות. יש כאן רוחניות שקטה שאינה מתנתקת מהקרקע.

שיפוע השורה הישר והעובי הבינוני של הקו מעידים על יציבות רגשית מרשימה. הכותב ש